# Huấn luyện và Đánh giá Mô hình Phát hiện Gian lận

Mục tiêu của notebook này là xây dựng, tinh chỉnh và đánh giá một mô hình Machine Learning cốt lõi (**Random Forest**) để phát hiện các giao dịch gian lận. Quá trình này sẽ đi qua các bước từ xây dựng mô hình cơ sở, kiểm định chéo, tối ưu siêu tham số, điều chỉnh ngưỡng ra quyết định cho đến giải thích mô hình.

## 1. Khởi tạo và Nạp dữ liệu (Setup & Load Data)

Trong bước này, chúng ta sẽ nạp dữ liệu đã được tiền xử lý ở các bước trước. Dữ liệu huấn luyện đã được áp dụng kỹ thuật **SMOTE** (Synthetic Minority Over-sampling Technique) để giải quyết vấn đề mất cân bằng lớp (class imbalance) nghiêm trọng trong tập dữ liệu phát hiện gian lận.

Vai trò của 4 tập dữ liệu:
- `X_train_final.csv`: Tập đặc trưng (features) dùng để huấn luyện mô hình. Đã được chuẩn hóa và cân bằng lớp bằng SMOTE.
- `y_train.csv`: Nhãn (labels) tương ứng cho tập huấn luyện.
- `X_test_final.csv`: Tập đặc trưng dùng để kiểm thử độc lập. **Không bị tác động bởi quá trình SMOTE** nhằm phản ánh đúng phân phối tự nhiên của thực tế.
- `y_test.csv`: Nhãn thực tế của tập kiểm thử để đánh giá độ chính xác thực tế của mô hình.

In [ ]:
# Import các thư viện phân tích dữ liệu và tính toán cốt lõi
import pandas as pd
import numpy as np

# Import các thư viện vẽ biểu đồ trực quan (Data Visualization)
import matplotlib.pyplot as plt
import seaborn as sns

# Import các module từ Scikit-Learn cho Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, f1_score
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

# Import thư viện để đóng gói và xuất mô hình
import joblib

# Thiết lập style cho biểu đồ đẹp mắt hơn
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Đường dẫn tương đối tới thư mục chứa dữ liệu
data_dir = '../data/'

# Đọc 4 tập dữ liệu từ các file CSV
X_train = pd.read_csv(data_dir + 'X_train_final.csv')
y_train = pd.read_csv(data_dir + 'y_train.csv').values.ravel() # flatten y thành mảng 1D cho tương thích mô hình
X_test = pd.read_csv(data_dir + 'X_test_final.csv')
y_test = pd.read_csv(data_dir + 'y_test.csv').values.ravel()

# In ra kích thước của các tập dữ liệu để kiểm tra tính hợp lệ
print(f"Kích thước tập huấn luyện (X_train, y_train): {X_train.shape}, {y_train.shape}")
print(f"Kích thước tập kiểm thử (X_test, y_test)  : {X_test.shape}, {y_test.shape}")

## 2. Huấn luyện Mô hình Cơ sở (Baseline Model)

Chúng ta lựa chọn **Random Forest** làm thuật toán cốt lõi cho hệ thống chống gian lận này vì các lý do thực tiễn sau:
1. **Khả năng xử lý dữ liệu dạng bảng (Tabular Data) xuất sắc**: Random Forest thường cho kết quả vượt trội trên tabular data so với các mô hình Neural Networks truyền thống mà không đòi hỏi chuẩn hóa quá phức tạp.
2. **Tính minh bạch (Explainability)**: Mô hình cung cấp khả năng truy xuất mức độ quan trọng của đặc trưng (Feature Importance), giúp giải thích cho nghiệp vụ biết tại sao giao dịch bị đánh dấu.
3. **Chống Overfitting hiệu quả**: Nhờ cơ chế Ensemble (học máy tập hợp) từ nhiều Decision Trees hoạt động độc lập.

Ta sẽ khởi tạo và đánh giá nhanh một mô hình với các tham số mặc định.

💡 **Lý thuyết về Random Forest:**

Random Forest là thuật toán Ensemble Bagging — kết hợp nhiều Decision Trees độc lập. Cơ chế này giúp giảm Variance (chống Overfitting) so với một Decision Tree đơn lẻ. Theo lý thuyết Bias-Variance Tradeoff: RF hi sinh một chút Bias để đổi lấy Variance thấp hơn đáng kể.

In [ ]:
# Khởi tạo mô hình Random Forest Classifier
# Sử dụng n_jobs=-1 để tận dụng toàn bộ số nhân CPU máy tính, giúp tăng tốc độ huấn luyện
# Thiết lập random_state=42 để đảm bảo kết quả đánh giá luôn giống nhau ở các lần chạy
rf_baseline = RandomForestClassifier(n_jobs=-1, random_state=42)

# Huấn luyện mô hình trên tập Train
print("Đang huấn luyện mô hình Cơ sở...")
rf_baseline.fit(X_train, y_train)
print("Huấn luyện hoàn tất!")

In [ ]:
# Chạy dự đoán trên tập kiểm thử (Test set)
y_pred_baseline = rf_baseline.predict(X_test)

# Đánh giá tổng quan thông qua Báo cáo phân loại
print("=== Báo cáo Phân loại (Classification Report) ===")
print(classification_report(y_test, y_pred_baseline))

# Lấy Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_baseline)

# Vẽ trực quan Ma trận nhầm lẫn bằng heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Bình thường (0)', 'Gian lận (1)'], 
            yticklabels=['Bình thường (0)', 'Gian lận (1)'])
plt.title('Ma trận Nhầm lẫn (Confusion Matrix) - Mô hình Cơ sở', fontsize=14, fontweight='bold')
plt.xlabel('Dự đoán của mô hình (Predicted Label)', fontsize=12)
plt.ylabel('Thực tế (True Label)', fontsize=12)
plt.show()

print("💡 NHẬN XÉT: Trong bài toán này, ta cần đặc biệt chú ý vào chỉ số Recall của class 1 (Gian lận). Nó thể hiện mô hình đã bắt được bao nhiêu % tổng số các vụ gian lận thực sự xảy ra.")

## 3. Xác thực chéo (K-Fold Cross Validation)

Để đảm bảo rằng kết quả đánh giá hiện tại không phải do sự may mắn tình cờ trong quá trình chia tập Train/Test ngẫu nhiên, ta cần thực hiện **Cross Validation** (Xác thực chéo). Kỹ thuật này giúp tránh **Overfitting** và đánh giá năng lực tổng quát hóa của mô hình một cách khách quan nhất.

Quá trình **5-Fold CV** sẽ chia tập Train thành 5 phần (fold) bằng nhau, lần lượt dùng 1 phần để Validate và 4 phần còn lại để Train. Điểm số cuối cùng là trung bình của 5 lần kiểm tra này.

In [ ]:
# Số lượng Folds = 5
cv_folds = 5

print(f"Thực hiện {cv_folds}-Fold Cross Validation trên tập Train, xin vui lòng đợi...")
# Thực hiện Cross Validation, ở đây ta theo dõi thước đo F1-Score (Cân bằng giữa Precision và Recall)
cv_scores = cross_val_score(rf_baseline, X_train, y_train, cv=cv_folds, scoring='f1', n_jobs=-1)

print("\n=== Kết quả Xác thực chéo (Cross Validation) ===")
print(f"Mảng điểm F1 của từng Fold : {np.round(cv_scores, 4)}")
print(f"Điểm F1 Trung bình         : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

### 3B. Learning Curve (Đường cong học tập)

Đánh giá hiện tượng Overfitting/Underfitting của mô hình thông qua Learning Curve.

In [ ]:
from sklearn.model_selection import learning_curve

# Tính toán Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    estimator=rf_baseline,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3,
    scoring='f1',
    n_jobs=-1
)

# Tính mean và std
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
val_scores_mean = np.mean(val_scores, axis=1)
val_scores_std = np.std(val_scores, axis=1)

# Vẽ biểu đồ
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color='blue', label='Train Score (F1)')
plt.plot(train_sizes, val_scores_mean, 'o-', color='orange', label='Validation Score (F1)')
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1, color='orange')

plt.title('Learning Curve - Random Forest Baseline', fontsize=14, fontweight='bold')
plt.xlabel('Số lượng mẫu huấn luyện (Training Samples)', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.legend(loc='best')
plt.grid(True)
plt.show()

# Tự động phân tích
gap = train_scores_mean[-1] - val_scores_mean[-1]
final_val = val_scores_mean[-1]

print("💡 NHẬN XÉT: Phân tích Learning Curve:")
print(f"- Khoảng cách (Gap) giữa Train và Val ở cuối: {gap:.4f}")
print(f"- Điểm Validation F1 ở cuối: {final_val:.4f}")

if gap > 0.1:
    print("⚠️ CẢNH BÁO: Mô hình có dấu hiệu Overfitting (Train score cao, Val score thấp, gap lớn).")
elif final_val < 0.7:
    print("⚠️ CẢNH BÁO: Mô hình có dấu hiệu Underfitting (Cả Train và Val score đều thấp).")
else:
    print("✅ KẾT LUẬN: Mô hình ổn định. Đường Train và Validation hội tụ tốt, không bị Overfitting nghiêm trọng hay Underfitting.")

## 4. Tối ưu Siêu tham số (Hyperparameter Tuning)

Mô hình ở trên đang dùng thông số mặc định của thư viện. Thay vì chọn tham số theo cảm tính (chọn mò), ta sẽ sử dụng kỹ thuật **`RandomizedSearchCV`** để tìm kiếm cấu hình tốt nhất.

Kỹ thuật này sẽ lấy mẫu ngẫu nhiên các tổ hợp cấu hình (như số lượng cây, độ sâu tối đa) và đánh giá chúng, giúp ta tiết kiệm thời gian đáng kể so với GridSearch mà vẫn tìm được mô hình tối ưu.

In [ ]:
# SO SÁNH GRID SEARCH vs RANDOMIZED SEARCH
# ─────────────────────────────────────────────────────────────
# GridSearchCV:     Thử TẤT CẢ 108 tổ hợp (3×4×3×3)
#                   → Đảm bảo tìm được tốt nhất, nhưng rất chậm
#
# RandomizedSearchCV: Chỉ thử n_iter=10 tổ hợp ngẫu nhiên (~9.3%)
#                   → Nhanh hơn nhiều, vẫn tìm được cấu hình tốt
#
# Trong bài toán này ta chọn RandomizedSearchCV vì:
#   1. Không gian 108 tổ hợp × CV × training time = quá chậm
#   2. Nghiên cứu cho thấy random search thường tìm được cấu hình
#      gần optimal chỉ với 10-20% không gian tìm kiếm
# ─────────────────────────────────────────────────────────────

# Định nghĩa không gian tìm kiếm (Grid) các siêu tham số
param_dist = {
    'n_estimators': [100, 200, 300],                # Số lượng cây quyết định (forest size)
    'max_depth': [None, 10, 20, 30],                # Độ sâu tối đa của mỗi cây (ngăn overfitting)
    'min_samples_split': [2, 5, 10],                # Số mẫu tối thiểu để chẻ một node
    'min_samples_leaf': [1, 2, 4]                   # Số mẫu tối thiểu tại một node lá cuối cùng
}

# Khởi tạo RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,             # Thử nghiệm ngẫu nhiên 10 cấu hình tổ hợp
    cv=3,                  # Sử dụng 3-Fold CV để kiểm tra (rút ngắn thời gian)
    scoring='f1',          # Mục tiêu là tối đa hóa điểm F1
    n_jobs=-1,             # Tận dụng tối đa CPU
    random_state=42,
    verbose=1
)

print("Bắt đầu quá trình Dò tìm Siêu tham số (Hyperparameter Tuning)...")
random_search.fit(X_train, y_train)

# Xuất kết quả siêu tham số tốt nhất
print("\n[HOÀN TẤT] Quá trình dò tìm đã xong!")
print("Các siêu tham số tốt nhất (Best Parameters) được tìm thấy:")
print(random_search.best_params_)

# Lưu lại mô hình tốt nhất để dùng cho các bước phân tích sau
rf_best_model = random_search.best_estimator_

## 5. Tinh chỉnh Ngưỡng quyết định (Decision Threshold Tuning)

❗ **RẤT QUAN TRỌNG TRONG NGHIỆP VỤ:** 

Trong thực tiễn chống gian lận (Fraud Detection), **chi phí của việc bỏ lọt một giao dịch gian lận (False Negative - Báo cáo thiếu) thường gây thiệt hại tiền bạc lớn hơn rất nhiều so với việc cảnh báo nhầm một giao dịch bình thường (False Positive)**.

Mặc định, mô hình ML sẽ báo gian lận nếu xác suất `P > 0.5`. Tuy nhiên, ta hoàn toàn có thể chủ động **hi sinh một chút Precision** (chấp nhận bắt nhầm một vài khách hàng bình thường, sau đó điều hướng họ qua luồng xác minh 2FA) để tối đa hóa **Recall** (đảm bảo tóm gọn hầu hết tội phạm mạng: thà bắt nhầm còn hơn bỏ sót).

Chúng ta sẽ vẽ biểu đồ **Precision-Recall Curve** và thiết lập một ngưỡng thấp hơn.

In [ ]:
# Dùng predict_proba để lấy XÁC SUẤT của class 1 (Gian lận) thay vì label cứng 0/1
y_scores = rf_best_model.predict_proba(X_test)[:, 1]

# Tính toán list điểm Precision, Recall ở toàn bộ các ngưỡng (thresholds) liên tục
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

# Vẽ biểu đồ đánh đổi Precision-Recall
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], 'b--', label='Precision (Độ chính xác)', linewidth=2)
plt.plot(thresholds, recalls[:-1], 'g-', label='Recall (Độ phủ)', linewidth=2)
plt.xlabel('Ngưỡng quyết định (Decision Threshold)', fontsize=12)
plt.ylabel('Tỷ lệ (Score %)', fontsize=12)
plt.title('Biểu đồ Đánh đổi Precision vs Recall theo Threshold', fontsize=15, fontweight='bold')
plt.axvline(x=0.3, color='red', linestyle=':', label='Ngưỡng đề xuất (0.3)')
plt.legend(loc='lower left', fontsize=12)
plt.grid(True)
plt.show()

In [ ]:
# Thiết lập ngưỡng quyết định mới tùy chỉnh (Custom Threshold)
CUSTOM_THRESHOLD = 0.3

# Áp dụng luật mới: Bất cứ giao dịch nào có xác suất gian lận >= 30% đều bị hệ thống gắn cờ gian lận
y_pred_custom = (y_scores >= CUSTOM_THRESHOLD).astype(int)

# Đánh giá lại với ngưỡng mới
print(f"=== Đánh giá mô hình với Custom Threshold = {CUSTOM_THRESHOLD} ===")
print(classification_report(y_test, y_pred_custom))

cm_custom = confusion_matrix(y_test, y_pred_custom)

# Vẽ lại Confusion Matrix
plt.figure(figsize=(7, 5))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Bình thường (0)', 'Gian lận (1)'], 
            yticklabels=['Bình thường (0)', 'Gian lận (1)'])
plt.title(f'Confusion Matrix (Threshold = {CUSTOM_THRESHOLD})', fontsize=14, fontweight='bold')
plt.xlabel('Dự đoán (Predicted Label)', fontsize=12)
plt.ylabel('Thực tế (True Label)', fontsize=12)
plt.show()

print("💡 NHẬN XÉT SỰ CẢI THIỆN:")
print(f"Khi hạ Threshold xuống {CUSTOM_THRESHOLD}, ta thấy số lượng False Negative đã giảm đi đáng kể (Recall tăng).")
print("Tuy nhiên, số lượng False Positive lại tăng lên. Đây là sự đánh đổi chấp nhận được đối với nghiệp vụ bảo mật ngân hàng.")

### 5B. ROC-AUC và PR-AUC

Tính toán và so sánh chỉ số ROC-AUC với PR-AUC.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc

# Tính ROC-AUC và PR-AUC
roc_auc = roc_auc_score(y_test, y_scores)
pr_auc = auc(recalls, precisions)

# Tính ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_scores)

# Tỷ lệ class 1 để vẽ baseline cho PR Curve
baseline_pr = len(y_test[y_test == 1]) / len(y_test)

# Vẽ 2 biểu đồ cạnh nhau
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Trái: ROC Curve
axes[0].plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Baseline')
axes[0].set_title('ROC Curve (FPR vs TPR)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Phải: Precision-Recall Curve
axes[1].plot(recalls, precisions, color='green', lw=2, label=f'PR Curve (AUC = {pr_auc:.4f})')
axes[1].axhline(y=baseline_pr, color='gray', lw=2, linestyle='--', label=f'Baseline (y={baseline_pr:.4f})')
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].legend(loc='upper right')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Tổng hợp Metrics:")
print(f"- ROC-AUC = {roc_auc:.4f}")
print(f"- PR-AUC  = {pr_auc:.4f}")

print("\n💡 NHẬN XÉT: Tại sao PR-AUC quan trọng hơn ROC-AUC trong bài toán này?")
print("Khi dữ liệu mất cân bằng nghiêm trọng (rất ít ca gian lận), ROC-AUC thường bị thổi phồng do số lượng True Negatives (TN) rất lớn làm FPR luôn giữ ở mức thấp. Ngược lại, PR-AUC chỉ tập trung vào class thiểu số (Precision và Recall), nên nó phản ánh chính xác hơn hiệu năng thực tế của hệ thống chống gian lận.")

## 6. Giải thích Mô hình và Đóng gói (Explainable AI & Export)

Để tạo niềm tin cho hội đồng, người quản lý và các chuyên viên nghiệp vụ, mô hình AI không nên hoạt động như một "hộp đen" (Black Box). Ta cần chứng minh được mô hình học được gì từ dữ liệu.

Đối với Random Forest, ta có thể phân tích mức độ đóng góp của từng biến số học thông qua thuộc tính **Feature Importance**. Bước cuối cùng, mô hình tối ưu nhất sẽ được serialize và lưu trữ dưới dạng nhị phân (`.pkl`) để tích hợp vào Backend API.

In [ ]:
# Trích xuất thuộc tính Mức độ quan trọng của đặc trưng từ mô hình đã Tuning
importances = rf_best_model.feature_importances_

# Ghép nối với danh sách tên cột của X_train thành DataFrame
feature_names = X_train.columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sắp xếp giảm dần và lấy Top 10 đặc trưng định đoạt mạnh nhất đến kết quả
top_10_features = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

# Vẽ biểu đồ thanh ngang (Horizontal Bar chart) cho trực quan
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=top_10_features, palette='magma')
plt.title('Top 10 Đặc trưng có tính chất Quyết định nhất (Feature Importances)', fontsize=15, fontweight='bold')
plt.xlabel('Mức độ ảnh hưởng (Importance Score)', fontsize=12)
plt.ylabel('Tên thuộc tính (Feature Name)', fontsize=12)
plt.show()

In [ ]:
import os

# Định nghĩa thư mục lưu trữ models
models_dir = '../models/'

# Tạo thư mục nếu chưa tồn tại trong source code
os.makedirs(models_dir, exist_ok=True)

# Cấu hình đường dẫn xuất file
model_path = os.path.join(models_dir, 'fraud_rf_model_tuned.pkl')

# Serialize và lưu mô hình xuống đĩa cứng bằng joblib
joblib.dump(rf_best_model, model_path)

print(f"✅ [THÀNH CÔNG] Đã lưu trữ toàn bộ cấu hình mô hình tốt nhất tại: {model_path}")
print("Mô hình này đã sẵn sàng để Backend FastAPI tải lên (load) phục vụ hệ thống dự đoán thời gian thực (Real-time Prediction)!")

## 7. So sánh Đa Mô hình (Model Comparison)

Đánh giá và so sánh mô hình Random Forest đã chọn với các thuật toán khác như Logistic Regression (Baseline tuyến tính) và Gradient Boosting (Ensemble Boosting) để củng cố quyết định lựa chọn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
import time

# Khởi tạo các mô hình cần so sánh
models = {
    'Logistic Regression (Linear Baseline)': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest (Tuned)': rf_best_model, # Mô hình đã tune (không cần train lại)
    'Gradient Boosting (Boosting)': GradientBoostingClassifier(random_state=42, n_estimators=100)
}

results = []

print("Đang tiến hành huấn luyện và thu thập metrics cho các mô hình...")

for name, model in models.items():
    start_time = time.time()
    
    # Huấn luyện mô hình (trừ RF đã tune)
    if name != 'Random Forest (Tuned)':
        model.fit(X_train, y_train)
        
    train_time = time.time() - start_time
    
    # Dự đoán
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else [0]*len(y_test)
    
    # Tính metrics
    f1 = f1_score(y_test, y_pred)
    # Tính confusion matrix để lấy recall và precision một cách an toàn
    cm_tmp = confusion_matrix(y_test, y_pred)
    recall = cm_tmp[1, 1] / (cm_tmp[1, 0] + cm_tmp[1, 1]) if (cm_tmp[1, 0] + cm_tmp[1, 1]) > 0 else 0
    precision = cm_tmp[1, 1] / (cm_tmp[0, 1] + cm_tmp[1, 1]) if (cm_tmp[0, 1] + cm_tmp[1, 1]) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'F1-Score': f1,
        'Recall': recall,
        'Precision': precision,
        'ROC-AUC': roc_auc,
        'Train Time (s)': train_time
    })

# Tạo DataFrame kết quả
results_df = pd.DataFrame(results).round(4)
print("\n=== Bảng So sánh Các Mô hình (Model Comparison) ===")
from IPython.display import display
display(results_df)

# Chuẩn bị dữ liệu vẽ biểu đồ Grouped Bar Chart
metrics_to_plot = ['F1-Score', 'Recall', 'Precision', 'ROC-AUC']

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('So sánh Hiệu năng giữa Các Mô hình', fontsize=16, fontweight='bold')

colors = ['skyblue', 'salmon', 'lightgreen']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors)
    ax.set_title(metric, fontsize=14)
    ax.set_ylim(0, 1.1)
    
    # Clean labels
    labels = ['LogReg', 'RF (Tuned)', 'GBM']
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=15)
    
    # Thêm giá trị số trên đầu mỗi cột
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("💡 NHẬN XÉT VÀ KẾT LUẬN LỰA CHỌN RANDOM FOREST:")
print("1. Với Tabular data (dữ liệu dạng bảng), Random Forest thường hoạt động rất tốt mà không cần tinh chỉnh quá nhiều so với Logistic Regression.")
print("2. Thuộc tính Feature Importance của RF rõ ràng và dễ giải thích hơn so với các mô hình Gradient Boosting, đáp ứng tốt yêu cầu nghiệp vụ ngân hàng.")
print("3. Cơ chế Bagging của RF giúp giảm Variance tự nhiên, chống Overfitting tốt hơn Boosting đối với những tập dữ liệu nhiễu như phát hiện gian lận.")

## 8. Phân tích Lỗi (Error Analysis)

Phân tích sâu vào các trường hợp dự đoán sai của mô hình, đặc biệt là các ca False Negative (Gian lận bị bỏ lọt) để hiểu giới hạn của hệ thống.

In [ ]:
# Tạo bản sao của X_test để phân tích lỗi
error_df = X_test.copy()
error_df['y_true'] = y_test
error_df['y_pred'] = y_pred_custom # Sử dụng kết quả với custom threshold (0.3)
error_df['fraud_proba'] = y_scores

# Phân loại kết quả dự đoán
def categorize_result(row):
    if row['y_true'] == 0 and row['y_pred'] == 0:
        return 'TN'
    elif row['y_true'] == 1 and row['y_pred'] == 1:
        return 'TP'
    elif row['y_true'] == 1 and row['y_pred'] == 0:
        return 'FN'
    else:
        return 'FP'

error_df['result_type'] = error_df.apply(categorize_result, axis=1)

# In thống kê số lượng
print(f"=== Thống kê phân loại kết quả (Threshold = {CUSTOM_THRESHOLD}) ===")
print(error_df['result_type'].value_counts())

# So sánh xác suất dự đoán trung bình giữa FN và TP
mean_proba_fn = error_df[error_df['result_type'] == 'FN']['fraud_proba'].mean()
mean_proba_tp = error_df[error_df['result_type'] == 'TP']['fraud_proba'].mean()

print("\nPhân tích ca bỏ lọt (False Negatives):")
print(f"- Xác suất gian lận trung bình của ca bị bỏ lọt (FN): {mean_proba_fn:.4f}")
print(f"- Xác suất gian lận trung bình của ca bắt đúng (TP)  : {mean_proba_tp:.4f}")

# Vẽ Histogram phân phối fraud_proba của FN
plt.figure(figsize=(10, 6))
fn_probas = error_df[error_df['result_type'] == 'FN']['fraud_proba']
sns.histplot(fn_probas, bins=20, color='red', kde=True)
plt.axvline(x=CUSTOM_THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Custom Threshold ({CUSTOM_THRESHOLD})')
plt.title('Phân phối Xác suất Dự đoán của các ca Gian lận Bị bỏ lọt (False Negatives)', fontsize=14, fontweight='bold')
plt.xlabel('Xác suất dự đoán gian lận (Fraud Probability)', fontsize=12)
plt.ylabel('Số lượng giao dịch', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

print("💡 NHẬN XÉT: Các ca False Negatives (FN) chủ yếu là các giao dịch 'borderline cases' có xác suất tập trung ngay dưới ngưỡng quyết định.")
print("Điều này cho thấy đây là giới hạn tự nhiên của mô hình phân loại (khó phân biệt rạch ròi), chứ không hẳn là lỗi của bản thân thuật toán. Các ca này có thể cần thêm các tính năng bổ sung (Feature Engineering) hoặc xem xét thủ công.")